# Unified plotting · 光电响应

复用 Integration-photonics 的统一绘图界面，只读读取数据，不连接仪器。
支持光电响应 JSON / SQLite、现有 SMU / SR830 归档和数值 CSV。

操作：选择数据目录 → Refresh records → 勾选数据源 → SQLite 再选择 Run ID → Load / refresh runs。
同一轮结果请选择 JSON 或 SQLite 中的一种，避免重复选择同一批观测。


In [ ]:
from pathlib import Path
import sys

PROJECT_ROOT = next((p for p in (Path.cwd().resolve(), *Path.cwd().resolve().parents)
                     if (p / 'src' / 'attodry_control').is_dir()), None)
if PROJECT_ROOT is None:
    raise RuntimeError('请在项目目录内打开 Notebook。')
SRC_PATH = str(PROJECT_ROOT / 'src')
existing = sys.modules.get('attodry_control')
if existing is not None and Path(existing.__file__).resolve().parent != Path(SRC_PATH) / 'attodry_control':
    raise RuntimeError('当前内核加载了其他项目，请重启内核后重新运行初始化。')
if SRC_PATH not in sys.path:
    sys.path.insert(0, SRC_PATH)
from IPython.display import display
from attodry_control.unified_plotting import UnifiedPlotDashboard
DATA_DIRECTORY = PROJECT_ROOT / 'run_data'


In [ ]:
dashboard = UnifiedPlotDashboard(DATA_DIRECTORY)
dashboard.add_plot('curve')
display(dashboard.widget)


## 功率与光生电压 / 电流

推荐选择实验输出子目录（例如 `run_data/nkt630_dark7`），以减少归档发现耗时。

- X：`actual.optical_power_w`（PM100D 实测 W）；Y：`measured.delta_voltage_v` 或 `measured.delta_current_a`。
- Add fixed-condition filter：`requested.optical_state = light`；多波长、多偏置或多栅压时继续筛选或分组。
- PM100D 闭环使用 NKT source_current / source_power 时，源百分比是调光参数，不必按 `target Optical source setting (%)` 分组；固定波长后 Stack/group 可选 None。
  源设定仍保留为可选诊断轴和导出字段；开环或无法确认反馈方式的数据继续检查该条件。已有内核更新代码后需重启该分析 Notebook 的内核并重新加载数据。
- 暗态 / 亮态读数：`measured.dark_voltage_v` / `measured.light_voltage_v`，电流对应 `_current_a`。
- 每行是一个条件的均值和差分。共享暗态不会被复制成独立观测；Mean ± SD / SEM 对这种单行结果为 n=1，误差未定义。
  `detector_sample_count` 和 `measured.detector_within_window_sd_*` 单独记录窗口内采样数和散布。
- 单暗点使用已记录的对应暗态，保留暗点 ID、时间间隔和未校正漂移标记；三窗口历史结果沿用原插值，不混用。
- 默认只显示整轮完成且清理通过的数据。查看失败记录时勾选 Include rejected/problem records；
  需要展示被标记的无效窗口时再选 Quality → AUDIT: include flagged。图及导出保留 AUDIT 标记；溢出码永不作为电压 / 电流绘制。
- 光功率是 `measurement_plane` 指定位置的读数，不自动解释为样品功率，也不自动计算 A/W 或 V/W。
- Export 保存 PNG / PDF / SVG、选中数据、统计数据和来源 manifest。历史原始文件不修改。
